# Cretona — Kaggle Wan2.1 worker

Truoc khi Run All:

1. **Settings -> Accelerator -> GPU T4 x2**
2. **Settings -> Internet -> On**
3. **Add-ons -> Secrets**: bat du 5 secret (xem cell duoi)
4. **Add Input**: upload `cretona-kaggle-worker-STEPS25.zip` lam private dataset

Notebook nay KHONG chua bat ky credential nao. Moi bi mat lay tu Kaggle Secrets.


## 1. Nap 5 Kaggle Secrets

| Secret | Gia tri |
|---|---|
| `CRETONA_WORKER_DATABASE_URL` | Session-pooler PostgreSQL URL cua `cretona_kaggle_worker`, co `sslmode=require` |
| `CRETONA_SUPABASE_URL` | `https://<project-ref>.supabase.co` |
| `CRETONA_SUPABASE_PUBLISHABLE_KEY` | Publishable key cua project (anon key cu cung duoc) |
| `CRETONA_WORKER_EMAIL` | Email tai khoan Auth danh rieng cho worker |
| `CRETONA_WORKER_PASSWORD` | Mat khau tai khoan do |

**Khong bao gio** them `service_role` hay URL admin cua backend vao Kaggle.


In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
for name in (
    "CRETONA_WORKER_DATABASE_URL",
    "CRETONA_SUPABASE_URL",
    "CRETONA_SUPABASE_PUBLISHABLE_KEY",
    "CRETONA_WORKER_EMAIL",
    "CRETONA_WORKER_PASSWORD",
):
    os.environ[name] = secrets.get_secret(name)

print("da nap", 5, "secret (khong in gia tri)")

# Audio is a feature flag, not a credential, so it does not belong in
# Kaggle Secrets. Unset it and the worker produces silent video exactly as
# it does today; that is the documented fallback, not a failure.
os.environ["CRETONA_WORKER_AUDIO"] = "1"
print("audio:", os.environ["CRETONA_WORKER_AUDIO"])


## 2. Xac minh va copy source tu private dataset

Hash duoi day tinh tren dang **LF** (blob git), khong phai file CRLF tren Windows.
Neu upload nham bundle cu, cell nay se dung lai — do la hanh vi dung.


In [ ]:
import hashlib
import shutil
from pathlib import Path

expected = {
    "backend/kaggle_worker.py":
        "04d3e70b78cd8a90c52644ac7d69e45ce75c58a98c95d52345bc0a9ad47c82ea",
    "backend/requirements-kaggle.txt":
        "46263bf0527742dde0d98a5da163afc1646a53b5f6de545ad3e94da43c99c669",
}
root = Path("/kaggle/working/Cretona")
for relative, expected_sha256 in expected.items():
    matches = list(Path("/kaggle/input").rglob(relative))
    assert len(matches) == 1, matches
    source = matches[0]
    actual = hashlib.sha256(source.read_bytes()).hexdigest()
    assert actual == expected_sha256, f'{relative}: {actual}'
    destination = root / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    print("OK", relative)


## 3. Smoke test re tien (khong nap model)

Hang doi rong thi thoat 0 va khong tai model. Chay de kiem tra Auth + ket noi
database truoc khi ton thoi gian cai dat.


In [ ]:
%cd /kaggle/working/Cretona
!python -m pip install -q "psycopg[binary]==3.3.4"
!python backend/kaggle_worker.py --once


## 4. Cai day du dependency

**Khong** cai hay nang cap `torch` — requirements co y dung ban CUDA Kaggle da co.


In [ ]:
!python -m pip install -q -r backend/requirements-kaggle.txt


## 4b. Cai MMAudio (chi khi bat audio)

Worker tim MMAudio o `/kaggle/working/MMAudio`. **Thieu buoc nay thi audio
khong bao gio chay** — `add_audio` chi ghi log rồi tra ve video im lang, job
van hoan tat binh thuong.

Bo qua cell nay neu khong bat `CRETONA_WORKER_AUDIO`.

> Checkpoint MMAudio la **CC-BY-NC 4.0**: dung cho do an duoc, thuong mai thi khong.

> Lan chay dau tien se tai ~6 GB weight, nen **job dau tien co audio lau hon
> khoang 3 phut** so voi cac job sau.


In [ ]:
%cd /kaggle/working
!git clone -q https://github.com/hkchengrex/MMAudio.git 2>/dev/null || echo 'da co san'
%cd /kaggle/working/MMAudio
!python -m pip install -q -e . 2>&1 | tail -3

# Khong duoc de pip doi torch: worker Wan2.1 chay tren dung ban CUDA Kaggle cap.
import torch
from pathlib import Path
print('torch      :', torch.__version__)
print('demo.py co :', Path('/kaggle/working/MMAudio/demo.py').is_file())

%cd /kaggle/working/Cretona


## 5. Chay worker

Cell nay chay lien tuc va phuc vu ca hang doi. De nguyen trong suot buoi demo.

- 25 denoising steps, 16 fps
- Clip 4 giay = 65 frame, uoc tinh ~20 phut moi clip tren 2xT4
- Chi xu ly 1 job mot luc

Muon chi chay dung 1 job roi dung thi them `--once`.


In [ ]:
!python backend/kaggle_worker.py
